# Explainability Analysis: Model Interpretability

This notebook demonstrates model explainability techniques:
- Grad-CAM heatmaps on spectrograms
- Attention weight visualization
- Segment-level predictions and localization
- Feature importance

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path

from src.pipeline import SlurringDetectionPipeline
from src.explainability.gradcam import GradCAM
from src.explainability.segment_localiser import SegmentLocaliser

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Load Pipeline and Sample Audio

In [ ]:
# Initialize pipeline
pipeline = SlurringDetectionPipeline()

# Load a sample audio file
train_df = pd.read_csv('../data/manifests/train.csv')
sample = train_df[train_df['label'] == 1].sample(n=1).iloc[0]  # Dysarthric sample
audio_path = sample['file_path']

print(f"Analyzing: {audio_path}")
print(f"True label: {'Dysarthric' if sample['label'] == 1 else 'Healthy'}")

## 2. Run Full Pipeline Analysis

In [ ]:
# Run analysis
result = pipeline.analyse(
    audio_file=Path(audio_path),
    patient_age=65,
    onset_hours=2.0,
)

print(f"\nPrediction Results:")
print(f"  Slurring Score: {result['slurring_score']}")
print(f"  Severity: {result['severity']}")
print(f"  Confidence: {result['confidence']:.4f}")
print(f"  Risk Score: {result['risk_score']}")
print(f"  Risk Tier: {result['risk_tier']}")

## 3. Visualize Grad-CAM Heatmap

In [ ]:
# Note: This requires the actual trained model
# For now, we'll visualize the concept

# Load audio and compute spectrogram
y, sr = librosa.load(audio_path, sr=16000)
mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
log_mel = librosa.power_to_db(mel_spec, ref=np.max)

# Create mock heatmap (replace with actual Grad-CAM output)
mock_heatmap = np.random.rand(*log_mel.shape) * 0.5
# In production: heatmap = gradcam.generate_heatmap(input_tensor, target_class)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Original spectrogram
librosa.display.specshow(log_mel, sr=sr, x_axis='time', y_axis='mel', ax=axes[0], cmap='viridis')
axes[0].set_title('Mel Spectrogram', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Mel Frequency')

# Grad-CAM overlay
librosa.display.specshow(log_mel, sr=sr, x_axis='time', y_axis='mel', ax=axes[1], cmap='gray', alpha=0.5)
librosa.display.specshow(mock_heatmap, sr=sr, x_axis='time', y_axis='mel', ax=axes[1], cmap='Reds', alpha=0.6)
axes[1].set_title('Grad-CAM Heatmap (Model Attention)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Mel Frequency')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 4. Segment Localization

In [ ]:
# Visualize segment annotations
if 'segments' in result and result['segments']:
    segments = result['segments']
    
    fig, ax = plt.subplots(figsize=(16, 6))
    
    # Plot waveform
    librosa.display.waveshow(y, sr=sr, ax=ax, alpha=0.6)
    
    # Overlay segments
    for segment in segments:
        start = segment.get('start_time', 0)
        end = segment.get('end_time', 0)
        severity = segment.get('severity', 'unknown')
        
        color_map = {'severe': '#e74c3c', 'moderate': '#f39c12', 'mild': '#f1c40f', 'none': '#2ecc71'}
        color = color_map.get(severity, '#95a5a6')
        
        ax.axvspan(start, end, alpha=0.3, color=color, label=severity)
    
    ax.set_title('Segment-Level Predictions', fontsize=14, fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Amplitude', fontsize=12)
    
    # Remove duplicate labels
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), fontsize=10)
    
    plt.tight_layout()
    plt.show()
else:
    print("No segments available in result")

## 5. Feature Importance (Mock Example)

In [ ]:
# Mock feature importance scores
# In production: Extract from model attention/gradients
feature_importance = {
    'F0 Variability': 0.25,
    'Speaking Rate': 0.22,
    'Vowel Space': 0.18,
    'Energy Modulation': 0.15,
    'Pause Ratio': 0.12,
    'MFCC Features': 0.08,
}

features = list(feature_importance.keys())
scores = list(feature_importance.values())

plt.figure(figsize=(10, 6))
plt.barh(features, scores, color='steelblue', edgecolor='black')
plt.xlabel('Importance Score', fontsize=12)
plt.title('Feature Importance for Dysarthria Detection', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Summary

Explainability techniques help clinicians:
- **Grad-CAM**: Shows which time-frequency regions triggered the prediction
- **Segment Localization**: Identifies specific portions of speech with slurring
- **Feature Importance**: Highlights which acoustic characteristics are most discriminative

This transparency builds trust and aids clinical decision-making.